# Microdrama Promo Finder (Kuku Assignment - Problem 1)

This notebook demonstrates the end-to-end Promo Finder pipeline:
1. **Parsing** 50 episodes with frame-accurate timestamps.
2. **Candidate Generation** (LLM or deterministic offline baseline).
3. **Validation & Snapping** (enforcing 20-60s window, hook verification, cliffhanger verification).
4. **Heuristic Feature Scoring** (pace, monologue penalty, action density, question cuts).
5. **Global Selection & Deduplication**.
6. **Multi-objective Ranking & Diversity Filtering**.

In [ ]:
import sys
from pathlib import Path
import os
import pandas as pd
import json

# Ensure promo-finder package is in python path
repo_path = Path("./promo-finder").resolve() if Path("./promo-finder").exists() else Path(".").resolve()
if str(repo_path) not in sys.path:
    sys.path.insert(0, str(repo_path))

from promo_finder.pipeline import run_pipeline
print("Promo Finder package imported successfully!")

## 1. Execute the Pipeline

Set `offline=False` and export `GEMINI_API_KEY` to run with Gemini, or run with `offline=True` for a deterministic baseline run.

In [ ]:
show_folder = repo_path / "data" / "show"
api_key_set = bool(os.environ.get("GEMINI_API_KEY"))
use_offline = not api_key_set

print(f"Running pipeline (offline={use_offline}, GEMINI_API_KEY set={api_key_set})...")

result = run_pipeline(
    input_folder=show_folder,
    run_id="notebook_run",
    output_dir=repo_path / "outputs",
    config_path=repo_path / "config.yaml",
    offline=use_offline,
    log_callback=print
)

output_dir = Path(result["output_path"])
print(f"\nPipeline completed. Results saved to: {output_dir}")

## 2. Inspect Validation & Parsing Results

In [ ]:
with open(output_dir / "validation_report.json") as f:
    val_data = json.load(f)

print("Validation Summary:")
print(json.dumps(val_data, indent=2))

## 3. View Top 10 Ranked Promos

In [ ]:
df = pd.read_csv(output_dir / "promos.csv")
cols = ["rank", "candidate_id", "duration_s", "hook_quote", "cliffhanger_quote", "final_blended_score"]
df[cols].head(10)

## 4. Read Top Promo Detail & Posting Rationale

In [ ]:
with open(output_dir / "ranked_report.md") as f:
    report_md = f.read()

# Print first 100 lines of markdown report
print("\n".join(report_md.splitlines()[:80]))